# 2020 311 hierarchy data prep to be used for SatScan

In [ ]:
import geopandas as gpd
import pandas as pd
df = gpd.read_file('../../data/2020/hierarchical_data_2020.geojson')


In [ ]:
df.head()

In [ ]:
camp_data = df[df["Camp"] == True]
camp_data.head()

In [ ]:
cols_to_drop = [
    "record_id",
    "Matches Homeless/Encampment Filter",
    "Vehicle",
    "Individual",
    "Active Camp",
    "Abandoned Camp",
    "Transient Camp",
    "Abandoned Vehicle",
    "Mental Health Crisis",
    "Neutral Presence",
    "Daily Activities",
    "Using Drugs"

]

camp_data = camp_data.drop(columns=cols_to_drop)


In [ ]:
camp_data.head()

In [ ]:
# Convert `date` column to datetime format
camp_data["date"] = pd.to_datetime(camp_data["date"]).dt.date
camp_data["date"] = pd.to_datetime(camp_data["date"])
camp_data

In [ ]:
# # Step 5: Save as a space-delimited text file
# camp_data_file_path = "../data/satscan_usable/camp_data_2020.csv"
# camp_data.to_csv(camp_data_file_path, index=False, header=True)

In [ ]:
# Ensure consistent location_id for duplicate geometries
# Step 1: Create a unique mapping of geometries to location_id
unique_geometries = camp_data[["geometry"]].drop_duplicates().reset_index(drop=True)
unique_geometries["location_id"] = range(1, len(unique_geometries) + 1)

# Step 2: Merge the location_id back into the original camp_data
camp_data = camp_data.merge(unique_geometries, on="geometry", how="left")

# Step 3: Add a unique caseID for each record
# camp_data["caseID"] = range(1, len(camp_data) + 1)

In [ ]:
camp_data

In [ ]:
# Group by location_id and count occurrences
location_counts = camp_data["location_id"].value_counts()

# Filter for location_ids with a count of 2 or more
duplicate_location_ids = location_counts[location_counts >= 2].index

# Filter camp_data for these location_ids
duplicate_records = camp_data[camp_data["location_id"].isin(duplicate_location_ids)]
duplicate_records

In [ ]:
# Remove full duplicates, keeping only the first occurrence
camp_data_deduplicated = camp_data.drop_duplicates(keep="first")

# Display the deduplicated DataFrame
camp_data_deduplicated


In [ ]:
# Optionally overwrite the original camp_data with the deduplicated version
camp_data = camp_data_deduplicated

In [ ]:
camp_data

In [ ]:
duplicate_geometries = camp_data[camp_data.duplicated(subset="location_id", keep=False)]
duplicate_geometries_sorted = duplicate_geometries.sort_values(by="location_id")

duplicate_geometries_sorted


In [ ]:
camp_data.head()

## Create cas file

In [ ]:
# Step 2: Aggregate cases by date and location_id
aggregated_df = camp_data.groupby(["date", "location_id"]).size().reset_index(name="cases")


In [ ]:
aggregated_df.head()

In [ ]:

# Step 3: Add a unique case_id starting from 1
aggregated_df = aggregated_df.sort_values(by=["date", "location_id"])  # Sort by date and location_id
aggregated_df["case_id"] = range(1, len(aggregated_df) + 1)  # Create unique case IDs


In [ ]:
aggregated_df.head()

In [ ]:
# Step 4: Reorder columns to match .cas file format
cas_df = aggregated_df[["case_id", "date", "cases", "location_id"]]


In [ ]:
cas_df.head()

In [ ]:
cas_df.shape

In [ ]:
records_with_multiple_cases = cas_df[cas_df["cases"] > 1]
records_with_multiple_cases

In [ ]:

# Step 5: Save as a space-delimited text file
cas_file_path = "../../data/satscan_usable/2020_camps.cas"
cas_df.to_csv(cas_file_path, index=False, header=True)



## Create geo file

In [ ]:
camp_data.dtypes

In [ ]:
from shapely.geometry import Point

# Ensure camp_data is a copy to avoid SettingWithCopyWarning
camp_data = camp_data.copy()

# Extract longitude and latitude directly from geometry objects
camp_data["longitude"] = camp_data["geometry"].apply(lambda point: point.x)
camp_data["latitude"] = camp_data["geometry"].apply(lambda point: point.y)

# Display updated camp_data
camp_data.head()

In [ ]:
# Step 2: Handle duplicate coordinates (same lat/lon -> same location_id)
geo_df = camp_data[["location_id", "latitude", "longitude"]].drop_duplicates(subset=["latitude", "longitude"])

geo_df.head()

In [ ]:
# Step 3: Validate coordinates
# Ensure latitude is between -90 and 90, and longitude is between -180 and 180
if not ((geo_df["latitude"].between(-90, 90)) & (geo_df["longitude"].between(-180, 180))).all():
    raise ValueError("Invalid coordinates found in the data.")

# Step 4: Sort by location_id
geo_df = geo_df.sort_values(by="location_id")
geo_df.head()


In [ ]:
geo_df.shape

In [ ]:
# Step 5: Save as a space-delimited text file
geo_file_path = "../../data/satscan_usable/2020_camps.geo"
geo_df.to_csv(geo_file_path, index=False, header=True)

In [ ]:
# Check for duplicate latitude and longitude combinations
duplicates = geo_df[geo_df.duplicated(subset=["latitude", "longitude"], keep=False)]

# Display the duplicate records
if not duplicates.empty:
    print("Duplicate latitude and longitude pairs found:")
    print(duplicates)
else:
    print("No duplicate latitude and longitude pairs found.")

In [ ]:
cas_df

In [ ]:
camp_data_counts = camp_data.groupby(['date', 'location_id']).size().reset_index(name='camp_data_count')
camp_data_counts

In [ ]:
# Step 2: Merge the camp_data_counts with cas_df on both location_id and date
merged_df = pd.merge(camp_data_counts, cas_df, on=['date', 'location_id'], how='inner')

# Step 3: Check if the counts from camp_data match the 'cases' in cas_df
merged_df['match'] = merged_df['camp_data_count'] == merged_df['cases']

In [ ]:
merged_df

In [ ]:
all_match = merged_df[merged_df["match"] == False]
all_match

In [ ]:
# Step 4: Verify that all rows match (i.e., all 'match' values should be True)
all_matches = merged_df['match'].all()

# Display the result
if all_matches:
    print("All counts match!")
else:
    print("There are mismatches.")

In [ ]:
# If using your .cas file DataFrame where date is in YYYYMMDD format:
min_date = cas_df['date'].astype(str).min()
max_date = cas_df['date'].astype(str).max()

print(f"Study Period Start: {min_date}")
print(f"Study Period End: {max_date}")

# To calculate total number of days:
from datetime import datetime

# If dates are in YYYY-MM-DD format:
start_date = datetime.strptime(str(min_date), '%Y-%m-%d')
end_date = datetime.strptime(str(max_date), '%Y-%m-%d')

# # If dates are in YYYYMMDD format:
# start_date = datetime.strptime(str(min_date), '%Y%m%d')
# end_date = datetime.strptime(str(max_date), '%Y%m%d')

total_days = (end_date - start_date).days + 1  # +1 to include both start and end dates
print(f"Total days in study period: {total_days}")